<a href="https://colab.research.google.com/github/aounraza379/AutiSense-AI/blob/main/AutiSense_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq gradio openai-whisper gTTS
!apt install ffmpeg -y -q
!pip install -q faiss-cpu sentence-transformers

In [15]:
import os, time, whisper, pickle, numpy as np
import gradio as gr
from gtts import gTTS
from groq import Groq
from sentence_transformers import SentenceTransformer
import faiss

# ─────────────────────────────────────────
# SETUP
# ─────────────────────────────────────────
GROQ_API_KEY = "api_here"
client = Groq(api_key=GROQ_API_KEY)

print("Loading Whisper...")
stt_model = whisper.load_model("tiny")
print("✅ Whisper ready")

print("Loading memory model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Memory model ready")

# ─────────────────────────────────────────
# RAG MEMORY SYSTEM
# ─────────────────────────────────────────
MEMORY_FILE = "autisense_memory.pkl"   # Saves memories to Colab disk
EMBED_DIM = 384                         # all-MiniLM-L6-v2 output size

def load_memory():
    """Load existing memory from disk, or create fresh."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "rb") as f:
            data = pickle.load(f)
            print(f"✅ Loaded {len(data['texts'])} memories from disk")
            return data["index"], data["texts"]
    index = faiss.IndexFlatL2(EMBED_DIM)
    return index, []   # fresh index + empty text list

def save_memory_to_disk(index, texts):
    """Persist memory to Colab disk."""
    with open(MEMORY_FILE, "wb") as f:
        pickle.dump({"index": index, "texts": texts}, f)

def add_memory(text, feedback, scenario, index, texts):
    """
    Convert an interaction into a memory string and store it.
    Example memory: 'In shop scenario, user said: I want apple. Feedback: missing please.'
    """
    memory_str = f"In {scenario} scenario, user said: '{text}'. Coach feedback was: '{feedback}'."
    vector = embedder.encode([memory_str], convert_to_numpy=True)
    faiss.normalize_L2(vector)           # Normalize for better similarity
    index.add(vector)
    texts.append(memory_str)
    save_memory_to_disk(index, texts)
    print(f"💾 Memory saved: {memory_str}")

def retrieve_memory(user_text, index, texts, top_k=3):
    """
    Find the most relevant past memories for the current input.
    Returns a formatted string to inject into the AI prompt.
    """
    if index.ntotal == 0:                # No memories yet
        return "This is the user's first session. Start simple and encouraging."

    vector = embedder.encode([user_text], convert_to_numpy=True)
    faiss.normalize_L2(vector)

    k = min(top_k, index.ntotal)         # Don't request more than we have
    distances, indices = index.search(vector, k)

    retrieved = [texts[i] for i in indices[0] if i < len(texts)]

    if not retrieved:
        return "No relevant past memories found. Treat this as a fresh start."

    memory_block = "\n".join(f"- {m}" for m in retrieved)
    return f"Past session memories about this user:\n{memory_block}"

# Load memory on startup
memory_index, memory_texts = load_memory()

# ─────────────────────────────────────────
# SCENARIO PROMPT ENGINE — now includes memory
# ─────────────────────────────────────────
def get_scenario_prompt(stage, scenario, memory_context):
    base_rules = """
- Always stay in character.
- Use very simple, warm English only.
- Keep replies to 1-2 short sentences maximum.
- Guide the child gently — never correct harshly.
- If the child's message is unclear, make a friendly guess and continue.
- Never say you are an AI.
"""
    stage_rules = f"""
Current conversation stage: {stage}
- GREETING: Welcome them warmly and introduce yourself.
- REQUEST: Listen and respond to what they want or need.
- CLARIFICATION: Ask ONE simple follow-up question only.
- CLOSING: End the conversation warmly and positively.
"""
    roles = {
        "shop":       "You are Emily, a friendly shopkeeper in a cozy little store. Help the child practice buying things politely.",
        "teacher":    "You are Ms. Anna, a kind and patient school teacher. Encourage the child to ask questions and speak up confidently.",
        "restaurant": "You are a cheerful restaurant waiter. Help the child practice ordering food from a menu politely.",
        "friend":     "You are Alex, a friendly child the same age. You are playing together and having a casual fun conversation.",
    }
    role_desc = roles.get(scenario, roles["shop"])

    return f"""{role_desc}

{base_rules}

{stage_rules}

MEMORY CONTEXT — use this to personalize your response:
{memory_context}
If the user struggled before (e.g. forgot please), gently hint at it.
If the user improved, praise them warmly.
"""

# ─────────────────────────────────────────
# FEEDBACK ENGINE — unchanged from Phase 1
# ─────────────────────────────────────────
def get_feedback(text, stage):
    t = text.lower().strip()

    if stage == "GREETING":
        if not any(w in t for w in ["hi", "hello", "hey", "good morning", "good afternoon"]):
            return "💡 Try starting with a greeting like 'Hello!' or 'Hi!' 👋"

    if not any(w in t for w in ["please", "thank you", "thanks", "excuse me", "sorry"]):
        return "💡 Remember to use polite words like 'please' or 'thank you' 🙏"

    if len(t.split()) < 3:
        return "💡 Try saying a little more — make a full sentence! 😊"

    if stage == "CLARIFICATION":
        if "?" not in text:
            return "💡 Try asking a question here — like 'Can I have...?' ❓"

    return "✅ Great job communicating! Keep it up! ⭐"

# ─────────────────────────────────────────
# SESSION
# ─────────────────────────────────────────
class Session:
    def __init__(self):
        self.scenario = "shop"
        self.stage = "GREETING"
        self.history = []

    def reset(self):
        self.stage = "GREETING"
        self.history = []
        return [], "Session reset! Say 'Hello' to start 👋", None, self.stage

    def update_stage(self, text):
        t = text.lower()
        if any(w in t for w in ["bye", "thanks", "goodbye", "see you"]):
            self.stage = "CLOSING"
        elif any(w in t for w in ["want", "buy", "need", "order", "learn", "have", "get"]):
            self.stage = "CLARIFICATION"
        elif any(w in t for w in ["hello", "hi", "hey"]):
            self.stage = "REQUEST"

session = Session()

# ─────────────────────────────────────────
# TTS
# ─────────────────────────────────────────
def generate_speech(text):
    try:
        filename = f"voice_{int(time.time())}.mp3"
        gTTS(text=text, lang='en', slow=False).save(filename)
        return filename
    except Exception as e:
        print(f"TTS error: {e}")
        return None

# ─────────────────────────────────────────
# SCENARIO CHANGE
# ─────────────────────────────────────────
def change_scenario(choice):
    session.scenario = choice
    session.reset()
    labels = {
        "shop":       "🛒 Shop mode! Emily the shopkeeper is ready.",
        "teacher":    "📚 Classroom mode! Ms. Anna is ready.",
        "restaurant": "🍽️ Restaurant mode! Your waiter is ready.",
        "friend":     "🧒 Friend mode! Alex wants to play!",
    }
    return labels.get(choice, "Ready!")

# ─────────────────────────────────────────
# MAIN FLOW — now with memory
# ─────────────────────────────────────────
def handle(audio, history):
    global memory_index, memory_texts

    if not audio:
        return history, "🎤 Please speak into the microphone 😊", None, session.stage

    try:
        text = stt_model.transcribe(audio)["text"].strip()
    except Exception as e:
        print(f"Whisper error: {e}")
        return history, "⚠️ Could not process audio. Try again.", None, session.stage

    if len(text.split()) < 2:
        return history, "🔇 Didn't catch that. Try speaking a little louder 😊", None, session.stage

    print(f"User said: {text}")

    session.update_stage(text)

    # Get feedback
    feedback = get_feedback(text, session.stage)

    # Retrieve relevant memories
    memory_context = retrieve_memory(text, memory_index, memory_texts)
    print(f"🧠 Memory context: {memory_context}")

    # Build prompt with memory
    system_prompt = get_scenario_prompt(session.stage, session.scenario, memory_context)
    messages = [{"role": "system", "content": system_prompt}]
    for msg in history:
        messages.append({"role": msg["role"], "content": msg["content"]})
    messages.append({"role": "user", "content": text})

    # Call Groq
    try:
        res = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            max_tokens=100,
            temperature=0.7,
        )
        ai_reply = res.choices[0].message.content.strip()
    except Exception as e:
        print(f"Groq error: {e}")
        ai_reply = "That sounds great! Can you tell me a little more? 😊"

    # Save this interaction to memory
    add_memory(text, feedback, session.scenario, memory_index, memory_texts)

    audio_file = generate_speech(ai_reply)

    history.append({"role": "user",      "content": text})
    history.append({"role": "assistant", "content": ai_reply})

    return history, feedback, audio_file, session.stage

# ─────────────────────────────────────────
# GRADIO UI — unchanged from Phase 1
# ─────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 AutiSense AI: Social Skills Trainer")
    gr.Markdown("*A safe space to practice real-life conversations*")

    with gr.Row():
        with gr.Column(scale=2):
            chat = gr.Chatbot(type="messages", height=420, label="Conversation")
            mic = gr.Audio(sources=["microphone"], type="filepath", label="🎤 Tap to Speak")
            audio_playback = gr.Audio(autoplay=True, label="AI Response")

        with gr.Column(scale=1):
            scenario_select = gr.Dropdown(
                choices=["shop", "teacher", "restaurant", "friend"],
                value="shop",
                label="🎭 Choose Scenario"
            )
            feedback_label = gr.Label(
                value="Select a scenario and say Hello!",
                label="💬 Coaching Feedback"
            )
            stage_box = gr.Textbox(
                label="📍 Conversation Stage",
                value="GREETING",
                interactive=False
            )
            reset_btn = gr.Button("🔄 Reset Conversation", variant="secondary")

    scenario_select.change(change_scenario, inputs=scenario_select, outputs=feedback_label)
    mic.change(handle, [mic, chat], [chat, feedback_label, audio_playback, stage_box])
    reset_btn.click(session.reset, None, [chat, feedback_label, audio_playback, stage_box])

demo.launch(debug=True)

Loading Whisper...
✅ Whisper ready
Loading memory model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Memory model ready
✅ Loaded 6 memories from disk


/tmp/ipykernel_2115/861696813.py:255: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_2115/861696813.py:261: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chat = gr.Chatbot(type="messages", height=420, label="Conversation")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2325ca0d1c81ffb4c3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://2325ca0d1c81ffb4c3.gradio.live


In [14]:
import os
print("Memory file exists:", os.path.exists("autisense_memory.pkl"))

Memory file exists: True
